# Part 1: Python Tools for Machine Learning

**Course:** 2026 KMITL Data Analytics

This notebook covers NumPy, pandas, Matplotlib, Seaborn, SciPy and scikit-learn,
each with a numerical example before real code.


## Learning objectives

By the end of this notebook you can:

1. Create NumPy arrays and read shape, dimensions and data type.
2. Select values with indexing and slicing.
3. Do basic numerical operations on arrays.
4. Compute mean, minimum, maximum and standard deviation; explain population vs sample.
5. Create pandas `Series` and `DataFrame`.
6. Load a small dataset from in-memory CSV text and inspect it.
7. Select columns, filter rows and inspect missing values.
8. Draw line, scatter, bar and histogram charts with Matplotlib.
9. Draw a correlation heatmap with Seaborn.
10. Use SciPy for scientific calculations and point distances.
11. Load a scikit-learn dataset, split it, build a leakage-free pipeline, measure accuracy.


## Required imports

Colab usually includes these libraries; for local use, install the project
requirements first. `%matplotlib inline` shows charts inside the page.


In [ ]:
%matplotlib inline

import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.spatial import distance

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")
np.random.seed(42)

print("Libraries loaded.")


## Dataset introduction

Our main dataset is **student scores**, created inside the notebook (one row per
student), plus a weekly score example and the built-in Iris dataset.

| Column | Meaning |
|---|---|
| `student_id` | Student identifier. |
| `name` | Student's name. |
| `group` | Class group, `A` or `B`. |
| `hours_study` | Hours studied per week. |
| `math_score` | Math exam score, 0-100. |
| `science_score` | Science exam score, 0-100; one value is missing on purpose. |
| `english_score` | English exam score, 0-100. |
| `attendance_rate` | Share of classes attended, 0-1. |

The scores are made up for teaching; they do not describe real students and cannot
prove that studying causes higher scores.


### Loading a dataset from CSV text

A **CSV** file is plain text: each line is one row, commas separate values, and the
first line usually holds the column names.

```text
name,score
Anna,55
Ben,68
```

pandas reads this with `pd.read_csv`. We keep the text in memory with `io.StringIO`,
which behaves like a file, so the notebook stays self-contained.


In [ ]:
# The dataset is written as CSV text, then loaded into a pandas DataFrame.
# Using an in-memory text buffer means the notebook is self-contained.
csv_text = """student_id,name,group,hours_study,math_score,science_score,english_score,attendance_rate
1,Anna,A,2,55,60,58,0.80
2,Ben,A,4,68,72,65,0.90
3,Cara,A,6,75,78,80,0.95
4,Dan,B,1,45,50,48,0.70
5,Eve,B,3,62,58,60,0.85
6,Finn,B,5,80,84,79,0.92
7,Gina,A,7,88,90,86,0.98
8,Hugo,B,2,52,49,55,0.75
9,Iris,A,8,95,92,91,0.99
10,Jack,B,4,70,,72,0.88
"""

df = pd.read_csv(io.StringIO(csv_text))
df


The table has 10 rows and 8 columns. Jack's `science_score` is `NaN` ("not a
number"), a missing value; pandas reads an empty field as `NaN` automatically.


## NumPy: arrays of numbers

NumPy is the base library for numerical work in Python. Its main object is the
**array**, an ordered box of values; arrays are fast because all values share one data
type.


### Arrays, shape and dimensions

The **shape** gives how many values lie along each direction (axis); the **number of
dimensions** is how many directions there are.

**Small example.** `[10, 20, 30]` has 3 values in one direction, so shape `(3,)` and
1 dimension. The table

```text
[[10, 20],
 [30, 40]]
```

has 2 rows and 2 columns, so shape `(2, 2)` and 2 dimensions.

General form: `n` values along the first axis give `(n,)`; `r` rows and `c` columns
give `(r, c)`. Here `n`, `r`, `c` are whole numbers.


In [ ]:
# A one-dimensional array (a vector).
scores = np.array([55, 68, 75, 45, 62, 80, 88, 52, 95, 70])
print("scores:", scores)
print("shape:", scores.shape)
print("dimensions:", scores.ndim)
print("data type:", scores.dtype)

# A two-dimensional array (a matrix): 2 rows and 3 columns.
small_table = np.array([[1, 2, 3], [4, 5, 6]])
print("small_table shape:", small_table.shape)
print("small_table dimensions:", small_table.ndim)


`scores` has shape `(10,)` and 1 dimension; `small_table` `(2, 3)` and 2 dimensions.
`scores` is an integer type because all values are whole numbers; its exact size can
depend on the platform.


### Array indexing and slicing

Indexing picks one value; counting starts at 0, so the first is `a[0]`. Negative
indexes count from the end, so `-1` is the last. Slicing `start:stop` includes `start`
and stops before `stop`.

**Small example.** With `a = [10, 20, 30, 40]`:

- `a[0]` is `10` (first value).
- `a[-1]` is `40` (last value).
- `a[1:3]` is `[20, 30]` (indexes 1 and 2, not 3).

General form: `a[start:stop]` returns values from `start` up to but not including
`stop`.


In [ ]:
a = np.array([10, 20, 30, 40])
print("a[0] =", a[0])
print("a[-1] =", a[-1])
print("a[1:3] =", a[1:3])

# Indexing and slicing also work on the student score array.
print("first score:", scores[0])
print("last score:", scores[-1])
print("first three scores:", scores[:3])
print("middle four scores:", scores[3:7])


The first value is `55`, the last `70`. `scores[:3]` gives `[55, 68, 75]`;
`scores[3:7]` gives four values (indexes 3-6).


### Basic numerical operations

NumPy applies an operation to every value at once: an **element-wise** operation.

**Small example.** If `a = [1, 2, 3]`, then `a + 10` is `[11, 12, 13]` and `a * 2` is
`[2, 4, 6]`.

General form: for arrays `a` and `b` of the same shape, `a + b` adds values in the
same position.


In [ ]:
print("scores + 5:", scores + 5)
print("scores - 5:", scores - 5)
print("scores * 2:", scores * 2)
print("scores / 100 (fractions):", scores / 100)

# Add a small bonus to every math score.
bonus = np.full(len(scores), 5)
print("scores + bonus:", scores + bonus)

# A scientific-style calculation: the square root.
print("square root of 16:", np.sqrt(16))


Every value changed: adding `5` raised each score by 5. `np.sqrt(16)` is `4.0`,
because a square root is not always whole.


### Mean, minimum and maximum

The **mean** (average) adds all values and divides by their number.

**Small example.** For `4` and `8`, the sum is `12` and there are 2 values, so the
mean is `12 / 2 = 6`.

General form:

```text
mean = (x_1 + x_2 + ... + x_n) / n
```

Here `x_1 ... x_n` are the values, `n` the count, and `Σ` (sigma) means "add all of
them"; in short, `mean = (1/n) * Σ x_i`.

The **minimum** is the smallest value, the **maximum** the largest; for `[4, 8, 6]`
they are `4` and `8`.


### Standard deviation: population and sample

The **standard deviation** measures how spread out values are around the mean.

**Step-by-step small example.** Take `2, 4, 6`.

1. Mean: `(2 + 4 + 6) / 3 = 4`.
2. Deviations: `-2`, `0`, `2`.
3. Squares: `4, 0, 4`.
4. Sum of squares: `8`.
5. Population variance (`/ 3`): `8 / 3 ≈ 2.667`.
6. Sample variance (`/ (count - 1) = 2`): `8 / 2 = 4`.
7. Standard deviations: `sqrt(2.667) ≈ 1.633` (population), `sqrt(4) = 2.000`
   (sample).

**Why two versions?** The population version describes a complete group; the sample
version a part standing for a larger unseen group. Dividing by `n - 1` instead of `n`
makes the variance estimate unbiased for independent samples from one population
(Bessel's correction); the square root is not exactly unbiased.

General form, for values `x_1 ... x_n` with mean `μ` (mu):

```text
population variance = (1 / n) * Σ (x_i - μ)^2
sample variance     = (1 / (n - 1)) * Σ (x_i - μ)^2
standard deviation  = sqrt(variance)
```

Here `x_i` is one value, `μ` the mean, `n` the count, and `Σ` means "add over all
values".


In [ ]:
# Use the math scores as a NumPy array (no missing values here).
math_scores = df["math_score"].to_numpy()

print("count:", math_scores.size)
print("mean:", np.mean(math_scores))
print("minimum:", np.min(math_scores))
print("maximum:", np.max(math_scores))

# ddof=0 divides by n (population). ddof=1 divides by n-1 (sample).
pop_std = np.std(math_scores, ddof=0)
sample_std = np.std(math_scores, ddof=1)
print("population standard deviation:", pop_std)
print("sample standard deviation:", sample_std)

# The same numbers with pandas.
print("pandas mean:", df["math_score"].mean())
print("pandas sample std:", df["math_score"].std())


The mean math score is `69.0`, minimum `45`, maximum `95`. The sample standard
deviation is a little larger than the population one; pandas `.std()` returns the
sample value by default, matching `np.std(..., ddof=1)`.


In [ ]:
# Small checks that confirm the key calculations are correct.
# The mean of the ten math scores is 69.
assert abs(np.mean(math_scores) - 69.0) < 1e-9

# np.std with ddof=0 must equal pandas' population std.
assert abs(np.std(math_scores, ddof=0) - df["math_score"].std(ddof=0)) < 1e-9

# Check the small hand-worked standard-deviation example.
assert np.isclose(np.std([2, 4, 6], ddof=0), np.sqrt(8 / 3))
assert np.isclose(np.std([2, 4, 6], ddof=1), 2.0)

print("NumPy and pandas checks passed.")


## pandas: Series and DataFrame

pandas organizes data into labelled tables. A **Series** is one labelled column; a
**DataFrame** is a table of several Series sharing row labels.


### Series and DataFrame

A **Series** is one column of labelled values; a **DataFrame** is a table of several
Series sharing the same row labels.

**Small example.** `[10, 20, 30]` with labels `["a", "b", "c"]` becomes a Series; two
such Series join side by side into a DataFrame.

General form: `pd.Series(data, index=labels)` creates a Series, and
`pd.DataFrame(dictionary)` makes a DataFrame with one column per key.


In [ ]:
# A small Series with custom labels.
study_hours = pd.Series([2, 4, 6], index=["Anna", "Ben", "Cara"], name="hours_study")
print(study_hours)

# Build a DataFrame from a dictionary.
small_frame = pd.DataFrame({
    "hours_study": [2, 4, 6],
    "math_score": [55, 68, 75],
}, index=["Anna", "Ben", "Cara"])
small_frame


The Series shows `2, 4, 6` labelled `Anna`, `Ben`, `Cara`. The DataFrame puts two
Series side by side, so we can read a full row per student.


### Creating and loading datasets

We created `df` by loading CSV text. pandas can also build a DataFrame from a
dictionary, list of dictionaries or NumPy arrays; `df` is the table used below.


In [ ]:
print("shape (rows, columns):", df.shape)
print("column names:", list(df.columns))
print("data types:")
print(df.dtypes)
df.head()


`df.shape` is `(10, 8)`: 10 students, 8 columns. Numeric columns are `int64` or
`float64`; `name` and `group` are `object` or `str`, depending on the pandas version.


### Selecting columns and filtering rows

**Selecting** picks columns: `df["math_score"]` gives a Series,
`df[["math_score", "science_score"]]` a DataFrame.

**Filtering** picks rows matching a condition: `df[df["math_score"] > 70]` keeps rows
with math score above 70.

**Small example.** From the table

```text
name   math_score
Anna   55
Ben    68
Cara   75
```

`math_score > 60` keeps Ben and Cara.

General form: `df[condition]` returns rows where `condition` is `True`, such as
`df["column"] > value`.


In [ ]:
# Select one column (a Series).
math_column = df["math_score"]
print(math_column)

# Select three columns (a DataFrame).
print(df[["name", "math_score", "science_score"]].head())

# Filter rows: students with math score above 70.
high_math = df[df["math_score"] > 70]
print("students with math_score > 70:")
print(high_math[["name", "math_score"]])

# Filter with two conditions using & (and).
strong_students = df[(df["math_score"] >= 70) & (df["attendance_rate"] >= 0.90)]
print("strong students:")
print(strong_students[["name", "math_score", "attendance_rate"]])


Selecting one column returns a Series, two a DataFrame. `math_score > 70` keeps four
students. The two-condition filter keeps students scoring at least 70 in math and
attending at least 90% of classes; wrap each condition in parentheses with `&`.


### Missing-value inspection

Missing values are empty entries, shown as `NaN`. Count them with `df.isna().sum()`.

**Small example.** For `[10, NaN, 30]`, `.isna()` gives `[False, True, False]` and the
sum is `1`.

General form: `df.isna()` is `True` for every missing cell; `.sum()` counts them per
column.


In [ ]:
print("missing values per column:")
print(df.isna().sum())

# Show only the rows that contain a missing value.
rows_with_missing = df[df.isna().any(axis=1)]
print("rows with at least one missing value:")
rows_with_missing


Only `science_score` has a missing value, in Jack's row; all other columns are
complete. We keep it and remember it when computing statistics, because some functions
skip `NaN` and others do not.


## Charts with Matplotlib and Seaborn

A chart turns numbers into a picture:

- A **line plot** shows values in order, often over time.
- A **scatter plot** shows the relationship between two numbers.
- A **bar chart** compares a summary value across groups.
- A **histogram** shows how often values fall in each range.

Charts have titles, axis labels and legends when useful; Matplotlib gives direct
control, Seaborn adds statistical charts. For the line plot, imagine weekly practice
scores 50, 60, 58, 70 in weeks 1-4; connecting points makes sense because weeks are
ordered.


In [ ]:
plt.figure(figsize=(8, 4))
weeks = [1, 2, 3, 4]
weekly_scores = [50, 60, 58, 70]
plt.plot(weeks, weekly_scores, marker="o", color="tab:blue", label="Practice score")
plt.title("Practice Scores Over Four Weeks")
plt.xlabel("Week")
plt.xticks(weeks)
plt.ylabel("Practice score (0-100)")
plt.legend()
plt.grid(True)
plt.show()


**Reading the line plot.** The practice score rises from 50 to 70 overall, with a
small fall 60 to 58 in week 3. The line shows change over ordered weeks, not scores
between observations.


In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(df["hours_study"], df["math_score"], s=80, color="tab:orange", label="Students")
plt.title("Math Score vs Study Hours")
plt.xlabel("Hours studied per week")
plt.ylabel("Math score (0-100)")
plt.legend()
plt.grid(True)
plt.show()


**Reading the scatter plot.** Each dot is one student. Students who studied more
hours tend to have higher math scores in this made-up dataset; this is a pattern, not
proof that studying causes higher scores.


In [ ]:
group_means = df.groupby("group")["math_score"].mean()
print(group_means)

plt.figure(figsize=(6, 4))
plt.bar("A", group_means["A"], label="Group A", color="tab:blue")
plt.bar("B", group_means["B"], label="Group B", color="tab:green")
plt.title("Average Math Score by Group")
plt.xlabel("Group")
plt.ylabel("Average math score (0-100)")
plt.legend()
plt.grid(axis="y")
plt.show()


**Reading the bar chart.** Bar height is the average math score per group; group A
has the higher average here. A bar chart compares one summary value across a few
groups.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["math_score"], bins=5, color="tab:purple", edgecolor="white")
plt.title("Distribution of Math Scores")
plt.xlabel("Math score (0-100)")
plt.ylabel("Number of students")
plt.grid(axis="y")
plt.show()


**Reading the histogram.** Scores are grouped into 5 equal ranges, counting students
per range. Most scored between about 45 and 80; the shape looks flat because there are
only 10 students.


### Correlation

**Correlation** measures how strongly two numbers move together, from `-1` to `1`:
`1` means both rise together in a straight line, `-1` means one rises as the other
falls, and `0` means no straight-line relationship.

**Small example.** `(1, 2), (2, 4), (3, 6)` rise in a straight line, so `r = 1`;
`(1, 6), (2, 4), (3, 2)` fall, so `r = -1`. For the first set, means are 2 and 4,
deviations `[-1, 0, 1]` and `[-2, 0, 2]`, matching products sum to `4`, and dividing
by `3 - 1 = 2` gives sample covariance `2`; sample standard deviations are `1` and
`2`, so `r = 2 / (1 * 2) = 1`.

General form, the Pearson correlation is

```text
r = covariance(x, y) / (std(x) * std(y))
```

Here `x`, `y` are the columns, `covariance` measures how they change together, and
`std` is the standard deviation. Dividing by both removes units, so `r` is between
`-1` and `1`. Use one convention for covariance and both standard deviations; a
constant column has undefined correlation because its standard deviation is zero.


In [ ]:
numeric_columns = ["hours_study", "math_score", "science_score", "english_score", "attendance_rate"]
correlation = df[numeric_columns].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson correlation"})
plt.title("Correlation Heatmap of Student Features")
plt.xlabel("Feature")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


**Reading the heatmap.** Each cell shows the correlation between two features; the
diagonal is `1.00`. Warm (red) cells are positive; `hours_study` and `math_score` are
strongly related here. `science_score` has one missing value, so its correlations use
9 students. Correlation is a pattern, not a cause.


## SciPy: scientific calculations and distance

SciPy adds scientific tools on top of NumPy. `scipy.stats` gives summary statistics
and `scipy.spatial` gives distances.


### Scientific summary statistics

**Small example.** `scipy.stats.describe([2, 4, 6])` gives count `3`, minimum `2`,
maximum `6`, mean `4` and variance - the same values computed by hand.

General form: `stats.describe(values)` returns `nobs` (count), `minmax`, `mean` and
`variance`.


In [ ]:
description = stats.describe(math_scores)
print("number of observations:", description.nobs)
print("minimum and maximum:", description.minmax)
print("mean:", description.mean)
print("variance (sample):", description.variance)
assert np.isclose(description.variance, np.var(math_scores, ddof=1))


SciPy reports the same count (`10`), mean (`69.0`) and min/max (`45, 95`) as NumPy.
`stats.describe` uses `ddof=1` by default, so its variance is the sample version,
about `256.22`; the assertion checks it matches NumPy.


### Distance between two points

A **distance** measures how far apart two points are; each point is a list of numbers,
for example `(1, 2)` and `(4, 6)`.

**Small numerical example (Euclidean distance).** Subtract: `4 - 1 = 3`, `6 - 2 = 4`.
Square: `9`, `16`. Add: `25`. Square root: `sqrt(25) = 5`.

**Small numerical example (Manhattan distance).** Add absolute differences:
`|3| + |4| = 7`.

General form:

```text
Euclidean distance = sqrt( (x1 - y1)^2 + (x2 - y2)^2 + ... + (xk - yk)^2 )
Manhattan distance = |x1 - y1| + |x2 - y2| + ... + |xk - yk|
```

Here `x1 ... xk` and `y1 ... yk` are the two points' coordinates, and `k` is how many
each has.


In [ ]:
point_a = (1, 2)
point_b = (4, 6)
print("Euclidean distance:", distance.euclidean(point_a, point_b))
print("Manhattan distance:", distance.cityblock(point_a, point_b))
assert np.isclose(distance.euclidean(point_a, point_b), 5.0)
assert np.isclose(distance.cityblock(point_a, point_b), 7.0)

# Real example: distance between two students using hours_study and math_score.
student_1 = (df.loc[0, "hours_study"], df.loc[0, "math_score"])
student_4 = (df.loc[3, "hours_study"], df.loc[3, "math_score"])
print("student 1 point:", student_1)
print("student 4 point:", student_4)
print("distance between students:", distance.euclidean(student_1, student_4))


**Reading the distances.** Euclidean distance between `(1, 2)` and `(4, 6)` is `5.0`;
Manhattan is `7.0`. For the two students, distance uses only `hours_study` and
`math_score`; a larger distance means less similarity. Distance depends on units, so
scale features first.


## scikit-learn: datasets, preprocessing, models and metrics

The workflow has four steps:

1. **Load** a dataset into features `X` and target `y`.
2. **Split** it into training and test parts.
3. **Preprocess and train** a model with a `Pipeline`, so preprocessing is learned
   only from training data.
4. **Measure** the result with a metric such as accuracy.

We use the built-in **Iris** dataset: 150 flowers, 4 sepal/petal measurements each
(cm), 3 species; features are the measurements, the target the species.

**Preprocessing** prepares values for a model; `StandardScaler` centres each feature
and adjusts its spread. With training values `[2, 4, 6]`, the mean is 4 and the
population standard deviation is about 1.633, so 6 becomes `(6 - 4) / 1.633 ≈ 1.225`.
In general:

```text
scaled_value = (value - training_mean) / training_population_std
```

A constant feature uses scale 1 to avoid dividing by zero.

A **model** learns a rule from examples; logistic regression here classifies flower
species. `.fit()` learns from labelled examples; `.predict()` returns predicted
labels. A **metric** measures quality: 4 of 5 correct gives accuracy `4 / 5 = 0.8`
(80%). In general:

```text
accuracy = number_of_correct_predictions / number_of_predictions
```


### Train/test split and data leakage

We train on one part of the data and test on the unseen part, to show whether the
model **generalizes**.

**Data leakage** happens when test information leaks into training, for example
scaling with the whole dataset's mean and standard deviation before splitting: the
test values then influence training, so the result is no longer independent and may
look too good.

**Small example.** Training scores `[10, 20]`, test score `[1000]`: the overall mean is
`(10 + 20 + 1000) / 3 = 343.3`, which uses the test value; correct scaling uses only
the training mean `(10 + 20) / 2 = 15`.

The safe way is a `Pipeline`: the scaler is fitted on training data only, then applied
to the test data.


In [ ]:
# 1. Load the built-in dataset.
iris = load_iris()
X = iris.data
y = iris.target

print("feature names:", iris.feature_names)
print("target names:", iris.target_names)
print("X shape:", X.shape)

# 2. Split into training and test sets. stratify keeps the class balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. Build a pipeline: scale the features, then train logistic regression.
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=200, random_state=42)),
])
model.fit(X_train, y_train)

# 4. Predict and measure accuracy.
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("number of test flowers:", len(y_test))
print("test accuracy:", accuracy)


**Reading the result.** The model predicts most test flowers correctly, accuracy about
`0.91` - 41 of 45. The `Pipeline` scaler learned from training data only, so there is
no leakage. Exact results may vary slightly with library versions; one score does not
guarantee future performance.


In [ ]:
# The accuracy must be a number between 0 and 1.
assert 0.0 <= accuracy <= 1.0
# A reasonable model on Iris should beat simple guessing (1/3).
assert accuracy > 0.33
print("scikit-learn checks passed. Accuracy:", round(accuracy, 3))


## Common mistakes

- **Confusing population and sample standard deviation.** Use `ddof=0` for a
  population, `ddof=1` for a sample; pandas `.std()` uses `ddof=1`, NumPy `np.std()`
  `ddof=0`.
- **Forgetting Python indexes start at 0.** The first value is `a[0]`, not `a[1]`.
- **Mixing up slicing and indexing.** `a[1:3]` returns two values; `a[1]` one.
- **Filtering a DataFrame with `and`/`or`.** Use `&` and `|`, wrapping each condition
  in parentheses.
- **Ignoring missing values.** Check `df.isna().sum()` first; some functions skip
  `NaN`, others do not.
- **Scaling before splitting.** This causes leakage; put the scaler in a `Pipeline`
  fitted on training data only.
- **Reading correlation as a cause.** Correlation is a pattern, not proof of cause.
- **Forgetting chart labels.** A chart without a title and axis labels is hard to
  read.


## Summary

In this notebook we used the main Python tools for machine learning:

- **NumPy** for arrays, shapes, dimensions, indexing, slicing and statistics.
- **Population vs sample standard deviation**: divide by `n` or `n - 1`.
- **pandas** for `Series`, `DataFrame`, loading CSV text, selecting, filtering and
  missing values.
- **Matplotlib** for line, scatter, bar and histogram charts.
- **Seaborn** for a correlation heatmap.
- **SciPy** for summary statistics and Euclidean and Manhattan distances.
- **scikit-learn** for loading Iris, splitting data, a leakage-free `Pipeline`, and
  accuracy.
